# Experiment 7: Dimensionality Reduction and Model Evaluation (With and Without PCA)
## Comprehensive Comparative Machine Learning Study
### Objectives:
1. Study the effect of dimensionality reduction using Principal Component Analysis (PCA) on 10 machine learning classifiers.
2. Train and validate models without PCA (original 30-dimensional feature space).
3. Train and validate models with PCA (reduced 10-dimensional feature space capturing >= 95% variance).
4. Perform systematic hyperparameter tuning and 5-fold Stratified Cross-Validation for both settings.
5. Record and analyze comparative performance metrics, ROC-AUC curves, confusion matrices, and stability across folds.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    StackingClassifier
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, classification_report
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 1. Dataset Ingestion and Exploratory Data Analysis

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print("Dataset shape:", X.shape)
print("Target counts:
", y.value_counts())
print("Benign (1):", (y == 1).sum(), f"({(y==1).mean()*100:.2f}%)")
print("Malignant (0):", (y == 0).sum(), f"({(y==0).mean()*100:.2f}%)")


## 2. Stratified Train-Test Split and Feature Standardization

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training shape:", X_train_scaled.shape)
print("Testing shape:", X_test_scaled.shape)


## 3. Principal Component Analysis (PCA) and Scree Plot

In [ ]:
pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_train_scaled)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)

n_components_95 = np.argmax(cum_var >= 0.95) + 1
var_explained_95 = cum_var[n_components_95 - 1] * 100

print(f"Optimal components for >= 95% variance: {n_components_95} (explaining {var_explained_95:.2f}%)")

pca = PCA(n_components=n_components_95, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)


## 4. Hyperparameter Tuning and 5-Fold Cross-Validation for All 10 Classifiers

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# List of 10 Evaluated Classifiers
# 1. SVM, 2. Naive Bayes, 3. KNN, 4. Logistic Regression, 5. Decision Tree,
# 6. Random Forest, 7. AdaBoost, 8. Gradient Boosting, 9. XGBoost, 10. Stacking
